# TARDIS_MODEL

## Importation des modules nécessaires

In [ ]:
import subprocess
import joblib
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

## Chargement de la donnée 

In [ ]:
# Vérification de l'existance du dataset clean
subprocess.run(["jupyter", "nbconvert", "--to", "notebook", "--execute", "tardis_eda.ipynb"], check=True)

# Lecture du dataset nettoyé
df = pd.read_csv("cleaned_dataset.csv", sep=";")

# Taille du dataset
print(f"Nombre d'échantillons: {df.shape[0]}")
print(f"Nombre de caractéristiques: {df.shape[1]}")

# Aperçu du dataset
print("\nAperçu du dataset:")
df.head()

In [ ]:
# Informations sur le dataset
print("Informations sur le dataset:")
df.info()

## Cleaning and Conversion

Cette partie ne devrait pas normalement exister. Mais au vue du retard des autres membres de l'équipe, je me vois obliger de faire un petit truc pour pouvoir tester des modèles. Il s'agira donc de sélectionner les colonnes numériques, de les convertir puis de supprimer les lignes contenant une ou plusieurs valeurs manquantes au niveau de ces colonnes.

In [ ]:
# Liste de quelques colonnes numériques
cols_numeriques = [
    "Number of scheduled trains",
    "Number of cancelled trains",
    "Number of trains delayed at departure",
    "Average delay of all trains at arrival"
]

# Conversion des colonnes en type numériques
for col in cols_numeriques:
    df[col] = pd.to_numeric(df[col], errors='coerce')

In [ ]:
# Reduction du nombre de colonnes
df = df[cols_numeriques]

# Taille du nouveau dataset
print(f"Nombre d'échantillons: {df.shape[0]}")
print(f"Nombre de caractéristiques: {df.shape[1]}")

# Aperçu du nouveau dataset
print("\nAperçu du nouveau dataset:")
df.head()

In [ ]:
# Nombre de valeurs manquantes
df.isnull().sum()

In [ ]:
# Suppression des valeurs manquantes
df = df.dropna()

In [ ]:
# Taille du dataset après suppression des valeurs manquantes
print(f"Nombre d'échantillons: {df.shape[0]}")
print(f"Nombre de caractéristiques: {df.shape[1]}")

## Choix des variables

In [ ]:
# Définition de X (features) et y (target)
X = df[["Number of scheduled trains",
        "Number of cancelled trains",
        "Number of trains delayed at departure"]]
y = df["Average delay of all trains at arrival"]

# Division en train et test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=123)

## Entrainement et Evaluation

In [ ]:
# Entrainement
model = LinearRegression()
model.fit(X_train, y_train)

# Prediction
y_pred = model.predict(X_test)

# Evaluation
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"RMSE\t:{rmse: .2f} minutes")
print(f"MAE\t:{mae: .2f} minutes")
print(f"R2\t:{r2: .4f}")

## Sauvegarde du modèle

In [ ]:
joblib.dump(model, "model.joblib")
print("Modèle sauvegardé avec succès")